# Experimentos del proyecto ECU 911: alta demanda de emergencias a t+3

Registra en MLflow una corrida por cada modelo de la Semana 10, con la preparación y la división por fecha importadas de `pipeline_proyecto.py`. Cada corrida guarda los hiperparámetros, cómo se dividieron los datos y la huella del CSV, la precisión promedio (AP) en entrenamiento y en prueba, los segundos de entrenamiento y el pipeline entrenado. Al final marca la corrida elegida como final, la carga desde el registro, comprueba que reproduce su AP y exporta la tabla de corridas a `corridas_proyecto.csv`.

La métrica es la AP porque solo el 8% de los casos es alta demanda: un modelo que no distingue nada saca la tasa de clase 1, y 1.0 es perfecto.

Para empezar de cero, borra las corridas del experimento `ecu911-alta-demanda-t3` en la interfaz de MLflow (pestaña **Ports**, puerto 5050), reinicia el kernel y corre todo.

In [1]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from pipeline_proyecto import CORTE, RUTA_DATOS, ReglaUmbral, cargar_y_dividir, construir_preparacion

# MLflow avisa que el modelo se guarda sin firma; importa al servirlo, no hoy.
logging.getLogger("mlflow").setLevel(logging.ERROR)

# p90_t3 entra en X solo para la regla de la linea base; la preparacion lo descarta.
X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir(columnas_extra=["p90_t3"])
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)
print(f"Tasa de clase 1  entrenamiento: {y_ent.mean():.3f}  prueba: {y_prueba.mean():.3f}")

Entrenamiento: (327264, 19) Prueba: (94976, 19)
Tasa de clase 1  entrenamiento: 0.081  prueba: 0.079


## La huella de los datos

In [2]:
def huella(ruta):
    """Los primeros 12 caracteres del hash MD5 del archivo: si cambia un
    solo byte del CSV, la huella cambia."""
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella(RUTA_DATOS)
print("Huella de", RUTA_DATOS.name, ":", huella_datos)

Huella de obt_emergencias_canton_dia.csv : 7624e6f357a7


## El experimento

In [3]:
# La direccion del servidor viene de MLFLOW_TRACKING_URI: el codigo no la escribe.
print("Registro de MLflow:", mlflow.get_tracking_uri())
EXPERIMENTO = "ecu911-alta-demanda-t3"
mlflow.set_experiment(EXPERIMENTO)

Registro de MLflow: http://127.0.0.1:5050


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1794194652293, experiment_id='3', last_update_time=1794194652293, lifecycle_stage='active', name='ecu911-alta-demanda-t3', tags={}>

## Entrenar y registrar

In [4]:
def evaluar(nombre, modelo, con_preparacion=True):
    """Entrena el modelo (despues de la preparacion, salvo la regla, que lee
    las columnas crudas), mide la AP en entrenamiento y en prueba, y registra
    la corrida en MLflow."""
    pasos = [("preparar", construir_preparacion())] if con_preparacion else []
    pipeline = Pipeline(pasos + [("modelo", modelo)])

    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({
            **modelo.get_params(),
            "division": "por fecha",
            "fecha_corte": str(CORTE.date()),
            "huella_datos": huella_datos,
            "filas_entrenamiento": len(X_ent),
            "filas_prueba": len(X_prueba),
        })

        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        ap_ent = average_precision_score(y_ent, pipeline.predict_proba(X_ent)[:, 1])
        ap_prueba = average_precision_score(y_prueba, pipeline.predict_proba(X_prueba)[:, 1])
        prediccion = pipeline.predict(X_prueba)

        mlflow.log_metrics({
            "ap_entrenamiento": ap_ent,
            "ap_prueba": ap_prueba,
            "brecha": ap_ent - ap_prueba,
            "precision_prueba": precision_score(y_prueba, prediccion, zero_division=0),
            "recall_prueba": recall_score(y_prueba, prediccion),
            "segundos": segundos,
        })
        mlflow.sklearn.log_model(pipeline, "modelo")

    print(f"{nombre:30s} AP entrenamiento {ap_ent:.4f}   prueba {ap_prueba:.4f}   ({segundos:.1f} s)")

## Líneas base

El dummy no usa ninguna columna: su AP es la tasa de clase 1, el piso. La regla del dominio dice "t+3 igual que t−4" (el último día que cayó el mismo día de la semana): alta demanda si ese día ya superó el P90.

In [5]:
evaluar("dummy (tasa de clase 1)", DummyClassifier(strategy="prior"))
evaluar("regla: t-4 supero el P90", ReglaUmbral(), con_preparacion=False)

dummy (tasa de clase 1)        AP entrenamiento 0.0808   prueba 0.0787   (0.7 s)


regla: t-4 supero el P90       AP entrenamiento 0.0902   prueba 0.1163   (0.0 s)


## Modelo lineal

Regresión logística con `class_weight="balanced"` por el desbalance. `C` es la regularización (más chico, más fuerte): pruebo una 100 veces más fuerte para ver si hace falta.

In [6]:
evaluar("logistica C=1", LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000))
evaluar("logistica C=0.01", LogisticRegression(C=0.01, class_weight="balanced", max_iter=2000))

logistica C=1                  AP entrenamiento 0.1537   prueba 0.2249   (5.7 s)


logistica C=0.01               AP entrenamiento 0.1537   prueba 0.2252   (3.5 s)


## Ensamblaje

El bosque sin límite se sobreajusta; el regularizado exige hojas de al menos 100 filas.

In [7]:
evaluar("bosque sin limite", RandomForestClassifier(
    n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42))
evaluar("bosque min_samples_leaf=100", RandomForestClassifier(
    n_estimators=100, min_samples_leaf=100, class_weight="balanced", n_jobs=-1, random_state=42))

bosque sin limite              AP entrenamiento 1.0000   prueba 0.1830   (13.6 s)


bosque min_samples_leaf=100    AP entrenamiento 0.2514   prueba 0.2327   (13.4 s)


## Red neuronal

Una capa oculta de 32 neuronas; `early_stopping` aparta el 10% del entrenamiento y se detiene cuando deja de mejorar ahí.

In [8]:
evaluar("red neuronal (32,)", MLPClassifier(
    hidden_layer_sizes=(32,), early_stopping=True, max_iter=200, random_state=42))

red neuronal (32,)             AP entrenamiento 0.2021   prueba 0.2425   (12.3 s)


## El registro, ordenado

In [9]:
corridas = mlflow.search_runs(experiment_names=[EXPERIMENTO], order_by=["metrics.ap_prueba DESC"])
columnas = ["run_id", "tags.mlflow.runName", "metrics.ap_entrenamiento", "metrics.ap_prueba",
            "metrics.brecha", "metrics.segundos", "params.fecha_corte", "params.huella_datos"]
print(corridas[columnas].round(4).to_string(index=False))

                          run_id         tags.mlflow.runName  metrics.ap_entrenamiento  metrics.ap_prueba  metrics.brecha  metrics.segundos params.fecha_corte params.huella_datos
5010e060545a4452b98af160808fe24a          red neuronal (32,)                    0.2021             0.2425         -0.0404           12.3290         2025-07-01        7624e6f357a7
a7c9689b915a4eb7ba39e5ba381ef6cb bosque min_samples_leaf=100                    0.2514             0.2327          0.0187           13.4326         2025-07-01        7624e6f357a7
c18423df439544498288ed6df61649bf            logistica C=0.01                    0.1537             0.2252         -0.0715            3.4627         2025-07-01        7624e6f357a7
2e03468dbe824fb783610c16ed38ae08               logistica C=1                    0.1537             0.2249         -0.0712            5.7372         2025-07-01        7624e6f357a7
6e1f836501cc4a309924ba23cb6f001a           bosque sin limite                    1.0000             0.1830

## La corrida final

Elijo el bosque regularizado, como en la Semana 10, Sesión 2: la red neuronal le gana por muy poco en AP de prueba, pero el bosque entrena más rápido y se puede explicar (la importancia de cada columna). Lo marco con `candidato` = `final`, lo cargo desde el registro y compruebo que reproduce su AP.

In [10]:
FINAL = "bosque min_samples_leaf=100"
fila_final = corridas[corridas["tags.mlflow.runName"] == FINAL].iloc[0]
run_id_final = fila_final["run_id"]
mlflow.MlflowClient().set_tag(run_id_final, "candidato", "final")

modelo_final = mlflow.sklearn.load_model("runs:/" + run_id_final + "/modelo")
ap_reproducida = average_precision_score(y_prueba, modelo_final.predict_proba(X_prueba)[:, 1])
print("Corrida final:", FINAL, run_id_final)
print(f"AP registrada: {fila_final['metrics.ap_prueba']:.10f}   reproducida: {ap_reproducida:.10f}")
print("Reproduce el registro:", abs(ap_reproducida - fila_final["metrics.ap_prueba"]) < 1e-9)

Corrida final: bosque min_samples_leaf=100 a7c9689b915a4eb7ba39e5ba381ef6cb
AP registrada: 0.2326863296   reproducida: 0.2326863296
Reproduce el registro: True


## La tabla de corridas

In [11]:
mlflow.search_runs(experiment_names=[EXPERIMENTO]).to_csv("corridas_proyecto.csv", index=False)
print("corridas_proyecto.csv:", len(corridas), "corridas")

corridas_proyecto.csv: 7 corridas
